In [ ]:
!pip install transformers datasets accelerate torch

In [ ]:
import os
print(os.listdir('/content'))

['.config', 'final_training_data.txt', 'sample_data']


In [ ]:
import os
print(os.listdir('/content'))

['.config', 'sample_data']


In [ ]:
import os
print(os.listdir('/content'))

['.config', 'final_training_data.txt', 'sample_data']


In [ ]:
with open("/content/final_training_data.txt", "r", encoding="utf-8") as f:
    data = f.read()

print("Characters:", len(data))
print("Words:", len(data.split()))
print("PROMPT count:", data.count("PROMPT"))

Characters: 61916
Words: 7645
PROMPT count: 74


In [ ]:
!rm -rf /root/.config/Google

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/Medical_Evidence_Brief_Generator"

In [ ]:
!cp /content/final_training_data.txt "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/"

In [ ]:
!ls "/content/drive/MyDrive/Medical_Evidence_Brief_Generator"

final_training_data.txt


In [ ]:
DATA_FILE = "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/final_training_data.txt"

with open(DATA_FILE, "r", encoding="utf-8") as f:
    data = f.read()

print("Characters:", len(data))
print("Words:", len(data.split()))
print("PROMPT count:", data.count("PROMPT"))

Characters: 61916
Words: 7645
PROMPT count: 74


In [ ]:
from datasets import Dataset

DATA_FILE = "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/final_training_data.txt"

with open(DATA_FILE, "r", encoding="utf-8") as f:
    data = f.read()

examples = data.split("PROMPT:")

texts = []

for example in examples:
    example = example.strip()
    if len(example) > 50:
        texts.append("PROMPT: " + example)

dataset = Dataset.from_dict({"text": texts})

print(dataset)
print("\nFirst example:\n")
print(dataset[0]["text"][:1000])

Dataset({
    features: ['text'],
    num_rows: 64
})

First example:

PROMPT: Semaglutide obesity outcomes

OUTPUT:

OVERVIEW:
Semaglutide is a GLP-1 receptor agonist used for chronic weight management in adults with obesity or overweight and weight-related comorbidities.

KEY EVIDENCE:
A systematic review and meta-analysis of four randomized controlled trials involving 3,613 participants without diabetes demonstrated an average weight reduction of 11.85% compared with placebo.

CLINICAL RELEVANCE:
The findings support semaglutide as an effective pharmacologic option for obesity management, particularly in patients who do not achieve adequate weight loss through lifestyle interventions alone.

SAFETY NOTES:
Common adverse events include nausea, vomiting, and other gastrointestinal symptoms. Increased rates of treatment discontinuation and serious gastrointestinal or hepatobiliary events were observed.

KEY TAKEAWAY:
Semaglutide produces clinically meaningful weight loss in obesity wit

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully!


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 64
})


In [ ]:
tokenized_dataset = tokenized_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/model"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Model folder created!")

Model folder created!


In [ ]:
print(tokenized_dataset)

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 64
})


In [ ]:
import transformers
print(transformers.__version__)

5.12.1


In [ ]:
import transformers

print(transformers.__version__)
print(transformers.__file__)

5.12.1
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [ ]:
from transformers import TrainingArguments

help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [ ]:
import inspect
from transformers import TrainingArguments

print(inspect.signature(TrainingArguments))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_compile_

In [ ]:
from transformers import TrainingArguments
import inspect

sig = inspect.signature(TrainingArguments)

print("overwrite_output_dir" in sig.parameters)
print(list(sig.parameters.keys())[:30])

False
['output_dir', 'per_device_train_batch_size', 'num_train_epochs', 'max_steps', 'learning_rate', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_steps', 'optim', 'optim_args', 'weight_decay', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'optim_target_modules', 'gradient_accumulation_steps', 'average_tokens_across_devices', 'max_grad_norm', 'label_smoothing_factor', 'bf16', 'fp16', 'bf16_full_eval', 'fp16_full_eval', 'tf32', 'gradient_checkpointing', 'gradient_checkpointing_kwargs', 'torch_compile', 'torch_compile_backend', 'torch_compile_mode', 'use_liger_kernel']


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Medical_Evidence_Brief_Generator/model",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    logging_steps=5,
    report_to=None
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Medical_Evidence_Brief_Generator/model",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    logging_steps=5,
    report_to=[]
)

print("TrainingArguments ready!")

TrainingArguments ready!


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Trainer ready!")

Trainer ready!


In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.012992
10,0.940378
15,0.842097
20,0.779470
25,0.742776
30,0.788509
35,0.836727
40,0.710764
45,0.579468
50,0.685898


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=640, training_loss=0.4007641075178981, metrics={'train_runtime': 131.5925, 'train_samples_per_second': 4.863, 'train_steps_per_second': 4.863, 'total_flos': 167226900480000.0, 'train_loss': 0.4007641075178981, 'epoch': 10.0})

In [ ]:
num_train_epochs=5

In [ ]:
import os

path = "/content/drive/MyDrive/Medical_Evidence_Brief_Generator"

for root, dirs, files in os.walk(path):
    print(root)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive"))

['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'Dr. Yashika Vardiyani - Resume (4).pdf', 'Dr. Yashika Vardiyani - Resume (3).pdf', 'Dr. Yashika Vardiyani - Resume (2).pdf', 'Dr. Yashika Vardiyani - Resume (1).pdf', 'Dr. Yashika Vardiyani - Resume.pdf', 'Job tracker 2026.gsheet', 'keyword research.gdoc', 'final_training_data.txt', 'Medical_Evidence_Brief_Generator']


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "model.safetensors" in files:
        print(root)

/content/drive/MyDrive/Medical_Evidence_Brief_Generator/model/checkpoint-500
/content/drive/MyDrive/Medical_Evidence_Brief_Generator/model/checkpoint-640


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/model/checkpoint-640"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
prompt = "PROMPT: Tirzepatide obesity outcomes\n\nOUTPUT:\n"

inputs = tokenizer(prompt, return_tensors="pt")

print(inputs)
print(inputs["input_ids"].shape)

{'input_ids': tensor([], size=(1, 0), dtype=torch.int64), 'attention_mask': tensor([], size=(1, 0), dtype=torch.int64)}
torch.Size([1, 0])


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer fixed")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer fixed


In [ ]:
prompt = "PROMPT: Tirzepatide obesity outcomes\n\nOUTPUT:\n"

inputs = tokenizer(prompt, return_tensors="pt")

print(inputs)
print(inputs["input_ids"].shape)

{'input_ids': tensor([[ 4805,  2662, 11571,    25, 39847,    89,   538,   265,   485, 13825,
         10906,   198,   198,  2606,  7250,  3843,    25,   198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 18])


In [ ]:
output = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

PROMPT: Tirzepatide obesity outcomes

OUTPUT:

OVERVIEW:
Diabetic obesity is a leading cause of morbidity and mortality among patients receiving oral tirzepatide.

KEY EVIDENCE:
Systematic review data indicate that tirzepatide has demonstrated substantial weight reduction compared with placebo, with no significant increase in major adverse events.

CLINICAL RELEVANCE:
Benefits extend to patients with and without type 2 diabetes, particularly those with multiple comorbidities.

SAFETY NOTES:
Weight reduction should be monitored alongside nutritional intake and lifestyle modifications.

KEY TAKEAWAY:
Tirzepatide is an effective pharmacologic option for reducing obesity-related comorbidities in diabetic patients.


In [ ]:
model.save_pretrained(
    "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/final_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/final_model"
)

print("Final model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved!


In [ ]:
import os

for root, dirs, files in os.walk(
    "/content/drive/MyDrive/Medical_Evidence_Brief_Generator/final_model"
):
    print(root)
    for f in files:
        print("  ", f)

/content/drive/MyDrive/Medical_Evidence_Brief_Generator/final_model
   config.json
   generation_config.json
   model.safetensors
   tokenizer_config.json
   tokenizer.json


In [ ]:
prompt = """PROMPT: Tirzepatide obesity outcomes

OUTPUT:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

output = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

PROMPT: Tirzepatide obesity outcomes

OUTPUT:

OVERVIEW:
Weight loss and insulin resistance are major contributors to the development and progression of obesity.

KEY EVIDENCE:
Meta-analyses of randomized controlled trials have demonstrated substantial weight loss and improved glycemic control compared with placebo.

CLINICAL RELEVANCE:
These findings may influence treatment selection and patient education regarding glycemic control.

SAFETY NOTES:
Both medications show promising clinical benefits in patients with type 2 diabetes who require insulin replacement.

KEY TAKEAWAY:
Tirzepatide consistently ranks among the most effective obesity-specific therapies currently available.


In [ ]:
topic = "Semaglutide obesity outcomes"

prompt = f"""PROMPT: {topic}

OUTPUT:
"""

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

PROMPT: Semaglutide obesity outcomes

OUTPUT:

OVERVIEW:
Early recognition and management of obesity-related comorbidities remain critical components of modern obesity management.

KEY EVIDENCE:
Systematic review evidence suggests that obesity-related comorbidities contribute significantly to morbidity and mortality in semaglutide users.

CLINICAL RELEVANCE:
Understanding comorbid obesity symptoms may reduce patient initiation and progression to symptomatic obesity.

SAFETY NOTES:
Although evidence suggests improvements in glycemic control, clinicians should monitor comorbid comorbid conditions for potential interactions.

KEY TAKEAWAY:
Emerging evidence highlights the important role of obesity comorbidity in managing obesity-related comorbidities.
